# CV Exploration: Catcross + Top-12 Interactions

**CV only — no auto-submit. Check OOF before deciding to use last slot.**

Combines two things that individually gave the best results:
- **Catcross**: C(6,2)=15 cross-cat pairs as CatBoost cat_features (LB 0.95382, best)
- **Top-12 interactions**: C(12,2)=66 pairwise TE products instead of C(8,2)=28

Hypothesis: catcross adds signal via CatBoost's ordered TE on joint categories;
more interaction features capture more numeric joint signals. Combined, they might compound.

Top-12 TE features (by |corr| with target) include all 8 cats + top-4 numerics.
Interaction count: C(12,2)=66 vs baseline C(8,2)=28.

Baseline: catboost_divye_fe_catcross OOF=0.95566  LB=0.95382

In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

TOP_N = 12   # <-- change to 10 or 13 to try variants

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}  TOP_N={TOP_N}')

Train: (630000, 13)  Test: (270000, 13)  TOP_N=12


In [2]:
def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq

def compute_oof_te(train_df, test_df, cols, y, skf):
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}
    for col in cols:
        key, tr_vals, te_vals, fold_test = f'{col}_te', train_df[col].values, test_df[col].values, []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_te[key][val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)
        test_te[key] = np.mean(fold_test, axis=0)
    return oof_te, test_te

def make_interaction_features(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}

def add_cross_cat_features(df, top_cols):
    df = df.copy()
    new_cols = []
    for c1, c2 in combinations(top_cols, 2):
        name = f'{c1}__x__{c2}'
        df[name] = df[c1].astype(str) + '_' + df[c2].astype(str)
        new_cols.append(name)
    return df, new_cols

def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


print('Frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('OOF target encoding...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

corrs  = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
ranked = sorted(corrs, key=corrs.get, reverse=True)
topN   = ranked[:TOP_N]
top_cat_6 = [col.replace('_te','') for col in ranked[:8] if col.replace('_te','') in CAT_FEATURES]

print(f'Top-{TOP_N} for interactions ({len(list(combinations(topN,2)))} pairs): {topN}')
print(f'Top-6 cats for cross-cat pairs: {top_cat_6}')

tr_inter = make_interaction_features(oof_te,  topN)
te_inter = make_interaction_features(test_te, topN)

Frequency encoding...
OOF target encoding...
Top-12 for interactions (66 pairs): ['thallium_te', 'chest_pain_type_te', 'max_hr_te', 'number_of_vessels_fluro_te', 'st_depression_te', 'exercise_angina_te', 'slope_of_st_te', 'sex_te', 'age_te', 'ekg_results_te', 'cholesterol_te', 'bp_te']
Top-6 cats for cross-cat pairs: ['thallium', 'chest_pain_type', 'number_of_vessels_fluro', 'exercise_angina', 'slope_of_st', 'sex']


In [3]:
CATBOOST_PARAMS = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    task_type='CPU', thread_count=-1, random_seed=42, verbose=0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    fold_tr_inter  = make_interaction_features(fold_tr_te,  topN)
    fold_val_inter = make_interaction_features(fold_val_te, topN)
    fold_te_inter  = make_interaction_features(fold_te_te,  topN)

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq, fold_tr_te,  fold_tr_inter)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_inter)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_inter)

    X_tr_aug,  cross_cols = add_cross_cat_features(X_tr_aug,  top_cat_6)
    X_val_aug, _          = add_cross_cat_features(X_val_aug, top_cat_6)
    X_te_aug,  _          = add_cross_cat_features(X_te_aug,  top_cat_6)

    all_cat = CAT_FEATURES + cross_cols

    m = CatBoostClassifier(**CATBOOST_PARAMS, cat_features=all_cat)
    m.fit(X_tr_aug, y_tr, eval_set=(X_val_aug, y_val), early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  '
          f'best_iter={m.best_iteration_}  n_features={X_tr_aug.shape[1]}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (catcross_top{TOP_N}): {cv_auc:.5f}')
print(f'catcross baseline:             0.95566  (top-8 interactions)')
print(f'Delta:                         {cv_auc - 0.95566:+.5f}')


=== Fold 1/5 ===
Fold 1  AUC=0.95602  best_iter=370  n_features=120

=== Fold 2/5 ===
Fold 2  AUC=0.95488  best_iter=410  n_features=120

=== Fold 3/5 ===
Fold 3  AUC=0.95574  best_iter=515  n_features=120

=== Fold 4/5 ===
Fold 4  AUC=0.95532  best_iter=543  n_features=120

=== Fold 5/5 ===
Fold 5  AUC=0.95619  best_iter=564  n_features=120

OOF AUC (catcross_top12): 0.95563
catcross baseline:             0.95566  (top-8 interactions)
Delta:                         -0.00003


In [4]:
# --- SAVE CSV (no submit) ---
# Run this cell manually only if OOF beats baseline and you want to use the submission slot.

import subprocess

label = f'catboost_divye_fe_catcross_top{TOP_N}'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')
print(f'OOF: {cv_auc:.5f}  (delta vs catcross: {cv_auc - 0.95566:+.5f})')
print()
print('To submit, run the cell below.')

Saved: submissions/catboost_divye_fe_catcross_top12.csv
OOF: 0.95563  (delta vs catcross: -0.00003)

To submit, run the cell below.


In [5]:
# --- SUBMIT — run manually only if you decide to use the slot ---

desc = f'{label} | cv_auc={cv_auc:.4f}'
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

catboost_divye_fe_catcross_top12: submitted
